# 20 Retention: the clocks, enforced

Raw files past their deletion date are removed and vault rows past expiry dropped, every action logged to `Files/retention_log.jsonl`. Scheduled daily; the Compliance page reads the same log.

In [ ]:
# The same wheel the app runs. pip through subprocess so the cell works in a scheduled run (%pip magics are interactive-only).
import subprocess, sys, traceback, datetime
LOG = "/lakehouse/default/Files/pipeline_log.txt"
def log(msg):
    with open(LOG, "a") as f: f.write(f"{datetime.datetime.utcnow():%H:%M:%S} {msg}\n")
open(LOG, "w").close()
# the wheel without its pins, plus only what the runtime lacks: pandas, pyarrow and Spark stay at the runtime's versions (Spark writes the Delta tables here)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "/lakehouse/default/Files/lib/atlas-0.1.0-py3-none-any.whl"], capture_output=True, text=True); log("wheel: " + (r.stderr[-300:] or "ok"))
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"python-docx pypdf pyyaml anthropic python-dotenv rapidfuzz duckdb presidio-analyzer".split()], capture_output=True, text=True); log("deps: " + (r.stderr[-400:] or "ok"))
r = subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_lg", "-q"], capture_output=True, text=True); log("spacy model: " + (r.stderr[-300:] or "ok"))
# the runtime image ships an old typing_extensions that survives an upgrade and is already imported by the kernel: replace it and reload
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "--force-reinstall", "--no-deps", "typing_extensions"], capture_output=True, text=True); log("typing_extensions: " + (r.stderr[-200:] or "ok"))
import importlib, typing_extensions; importlib.reload(typing_extensions); log(f"typing_extensions {typing_extensions.__version__ if hasattr(typing_extensions, '__version__') else ''} Sentinel={hasattr(typing_extensions, 'Sentinel')} at {typing_extensions.__file__}")
import pydantic; log(f"pydantic {pydantic.VERSION}")
import os
os.environ["ATLAS_ROOT"] = "/lakehouse/default"                 # Files/ for working data, Tables/ for the Delta serving tables
os.environ["ATLAS_CONF"] = "/lakehouse/default/Files/conf"
for k, v in dict(__ENV__).items(): os.environ[k] = v            # model endpoints and keys, substituted at deploy time; Key Vault in production
print(open(LOG).read())


In [ ]:
try:
    from atlas import retention
    print(retention.enforce(who="retention-job@fabric")); print(retention.report()["raw"]); log("retention ok")
except Exception:
    log(traceback.format_exc()); raise